# Radon post-detect raster lab

Reads a **manual label file**, pulls each label's **exact vectors** (via
`LabelEntry.vector_signatures` -> `label_schema.path_signature`), then renders each label's
vector cluster to a **raster** the same way `VectorClassification/parse.py` renders it right
before handing it to PaddleOCR's recognizer -- same dpi/padding rule
(`ocr_prep.render_cluster_with_dynamic_dpi`, `VectorClassification/config.py`'s own
`OCR_DPI`/`MIN_RENDER_SIDE_PX`/`MAX_RENDER_DPI`/`RENDER_PADDING_EXTRA_PT`). Each sample is
therefore a stand-in for "the crop as it looks right after PaddleOCR's detector has localized
it" -- a raster, not vector geometry.

For each rasterized sample, a real `skimage.transform.radon` **sinogram** is computed and
plotted next to the crop, purely to look at -- **tentative/experimental only**: no automatic
angle extraction, no scoring, no integration back into the pipeline yet.

## 0 - Config

In [ ]:
from pathlib import Path

import numpy as np

LABEL_PATH        = None     # None -> first (name-sorted) outputs/labels/*.json
PDF_PATH_OVERRIDE = None     # None -> LabelSet.pdf_path (resolved vs repo root)
PAGE_INDEX        = None     # None -> every page present in the label file

# A real Radon sinogram's projections repeat every 180 degrees, so this is a
# plain, independent full sweep -- not paired with a perpendicular companion
# the way a pure-vector deskew sweep would be.
SINOGRAM_THETA = np.arange(0.0, 180.0, 1.0)

## 1 - Path bootstrap

In [ ]:
import os, sys
_root = os.path.abspath(os.path.join('../..'))
if _root not in sys.path:
    sys.path.append(_root)

## 2 - Labels -> the exact labelled vector set per label

`extract_vectors(page)` once per page (raw `get_drawings()` geometry -- no classification,
no clustering, no rendering), keyed by `path_signature`; each label pulls its own vectors by
the signatures it stored.

In [ ]:
from rastervec.P1_Reading_Native.reader import Reader
from rastervec.P1_Reading_Native.vector_extract import extract_vectors
from rastervec.Evaluation.Labelling.label_schema import (
    load_labels, split_labelset_by_source, path_signature)
from rastervec.commons.helpers.geometry import union_bbox
from rastervec.commons.paths import output_dir, REPO_ROOT


def _resolve_pdf(raw):
    p = Path(str(raw).replace(chr(92), "/"))
    if p.is_file():
        return p
    for base in (Path.cwd(), REPO_ROOT):
        for cand in ((base / p), (base / p.name)):
            if cand.is_file():
                return cand.resolve()
    for sub in ("references", "references2"):
        cand = REPO_ROOT / sub / p.name
        if cand.is_file():
            return cand
    raise FileNotFoundError(f"cannot locate PDF {raw!r} (cwd={Path.cwd()}, repo={REPO_ROOT})")


def _pick_label_path():
    if LABEL_PATH:
        return Path(LABEL_PATH)
    cands = sorted(Path(output_dir("labels")).glob("*.json"))
    if not cands:
        raise FileNotFoundError("no outputs/labels/*.json; set LABEL_PATH")
    return cands[0]


label_path = _pick_label_path()
labels = load_labels(label_path)
pdf_path = _resolve_pdf(PDF_PATH_OVERRIDE or labels.pdf_path)
manual = split_labelset_by_source(labels)["manual"].entries
print(f"label file: {label_path}")
print(f"pdf:        {pdf_path}")
print(f"manual entries: {len(manual)}")

by_page = {}
for e in manual:
    if PAGE_INDEX is None or e.page_index == PAGE_INDEX:
        by_page.setdefault(e.page_index, []).append(e)

matched = []          # (entry, [Vector, ...])
with Reader(pdf_path) as r:
    for pidx in sorted(by_page):
        page_vectors = extract_vectors(r.get_page(pidx))
        sigmap = {path_signature(v): v for v in page_vectors}
        for e in by_page[pidx]:
            sel = [sigmap[s] for s in e.vector_signatures if s in sigmap]
            matched.append((e, sel))
            print(f"  p{pidx}  {e.text!r:12}  rot={e.expected_rotation:>4}  "
                  f"found {len(sel)}/{len(e.vector_signatures)} vectors")

print(f"\nmatched clusters: {len(matched)}")

## 3 - Render each sample the way OCR sees it (pre-recognition render + pad)

Same render/pad path `VectorClassification/parse.py` uses right before calling PaddleOCR's
detector/recognizer: `ocr_prep.render_cluster_with_dynamic_dpi` at `OCR_DPI`, dpi bumped up
(never down) for a small cluster, padded by half the cluster's own max stroke width plus
`RENDER_PADDING_EXTRA_PT` (`parse.py::_cluster_render_padding`).

In [ ]:
from rastervec.commons.renderer import ocr_prep
from rastervec.P3_Vector_Parsing.VectorClassification.config import (
    OCR_DPI, MIN_RENDER_SIDE_PX, MAX_RENDER_DPI, RENDER_PADDING_EXTRA_PT)


def _cluster_render_padding(vectors):
    """Mirrors `VectorClassification/parse.py::_cluster_render_padding`."""
    return max((v.width or 0.0) for v in vectors) / 2.0 + RENDER_PADDING_EXTRA_PT


samples = []   # dict(entry, vectors, image, gray, dpi_used, padding)
for e, vec in matched:
    if not vec:
        print(f"  {e.text!r:14} -- no vectors, skipped")
        continue
    padding = _cluster_render_padding(vec)
    image, dpi_used = ocr_prep.render_cluster_with_dynamic_dpi(
        vec, OCR_DPI, MIN_RENDER_SIDE_PX, MAX_RENDER_DPI, padding)
    gray = np.asarray(image.convert("L"), dtype=float)
    samples.append(dict(entry=e, vectors=vec, image=image, gray=gray,
                        dpi_used=dpi_used, padding=padding))
    print(f"  {e.text!r:14} rot={e.expected_rotation:>4}  "
          f"{len(vec)} vectors  render {gray.shape[1]}x{gray.shape[0]}px @ {dpi_used}dpi")

print(f"\nsamples: {len(samples)}")

## 4 - Sinogram (tentative, experimental)

Just a raw `skimage.transform.radon` call over each sample's raster -- no analysis of the
result yet, purely to look at what the transform produces on real post-detect-style crops.

In [ ]:
from skimage.transform import radon

for s in samples:
    s["sinogram"] = radon(s["gray"], theta=SINOGRAM_THETA, circle=False)

## 5 - Plot: raster crop + sinogram, per sample

In [ ]:
import matplotlib.pyplot as plt

for i, s in enumerate(samples):
    e = s["entry"]
    fig, (left, right) = plt.subplots(1, 2, figsize=(9, 4))
    fig.suptitle(f"#{i}  {e.text!r}   expected_rotation={e.expected_rotation}"
                 f"   ({len(s['vectors'])} vectors)", fontsize=10)

    left.imshow(s["gray"], cmap="gray")
    left.set_title(f"raster ({s['gray'].shape[1]}x{s['gray'].shape[0]}px @ {s['dpi_used']}dpi)",
                   fontsize=8)
    left.set_xticks([]); left.set_yticks([])

    sino = s["sinogram"]
    right.imshow(sino, cmap="gray", aspect="auto",
                extent=(SINOGRAM_THETA[0], SINOGRAM_THETA[-1], 0, sino.shape[0]))
    right.set_title("sinogram", fontsize=8)
    right.set_xlabel("theta (deg)"); right.set_ylabel("detector position (px)")

    plt.tight_layout(rect=[0, 0, 1, 0.92])
    plt.show()